In [170]:
import fitz  # PyMuPDF
import re
import json
import io
import gc
import sys
import os
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.document import DocumentStream
import requests
from bs4 import BeautifulSoup
import json
import time
import pandas as pd
import logging

# 1. Setup Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("scrape_logs.log", encoding='utf-8'),
        logging.StreamHandler() # This prints to jupyter cell output too
    ]
)
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully.")

# CONFIGURATION
PDF_PATH = r"C:/Users/Mohamed/Desktop/agent_juridique/data/juridique_files/code urbanisme.pdf"  
LAW_NAME = "code_urbanisme"
BATCH_SIZE = 6

print(f"✅ Ready to process: {PDF_PATH}")

✅ Libraries imported successfully.
✅ Ready to process: C:/Users/Mohamed/Desktop/agent_juridique/data/juridique_files/code urbanisme.pdf


## PDF INGESTOR

### Test Ingestion Strategy (OCR vs Standard)

In [171]:
doc = fitz.open(PDF_PATH)
is_locked = not (doc.permissions & fitz.PDF_PERM_COPY)
has_text = any(doc[i].get_text().strip() for i in range(min(3, len(doc))))

pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = False 

if is_locked and not has_text:
    print(f"🕵️ Strategy: FORCE OCR (Locked & No Text)")
    pipeline_options.do_ocr = True
    pipeline_options.ocr_options.force_full_page_ocr = True
elif not has_text:
    print(f"🕵️ Strategy: OCR (Scanned Image)")
    pipeline_options.do_ocr = True
else:
    print(f"🕵️ Strategy: STANDARD (Digital Text)")
    pipeline_options.do_ocr = False

converter = DocumentConverter(
    format_options={"pdf": PdfFormatOption(pipeline_options=pipeline_options)}
)
doc.close()

🕵️ Strategy: STANDARD (Digital Text)


### Convert a Single Batch to Raw Markdown

In [172]:
# Process pages 0 to BATCH_SIZE
doc = fitz.open(PDF_PATH)
batch_doc = fitz.open()
batch_doc.insert_pdf(doc, from_page=0, to_page=BATCH_SIZE-1)
pdf_bytes = batch_doc.tobytes()
batch_doc.close()

source = DocumentStream(name="test_batch.pdf", stream=io.BytesIO(pdf_bytes))
result = converter.convert(source)
md_text = result.document.export_to_markdown()

print("--- RAW MARKDOWN PREVIEW (First 1000 chars) ---")
print(md_text[:1000])

2026-04-19 18:18:56,118 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-04-19 18:18:56,120 - INFO - Going to convert document batch...
2026-04-19 18:18:56,121 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 38b1802ab226cd3dc476877290e6c7aa
2026-04-19 18:18:56,122 - INFO - Accelerator device: 'cpu'


2026-04-19 18:18:56,337 - INFO - HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-layout-heron/revision/main "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 6678.15it/s]
2026-04-19 18:18:56,687 - INFO - Processing document test_batch.pdf
2026-04-19 18:18:57,226 - ERROR - Stage preprocess failed for run 1, pages [6]: std::bad_alloc
2026-04-19 18:19:02,116 - INFO - Finished converting document test_batch.pdf in 6.00 sec.


--- RAW MARKDOWN PREVIEW (First 1000 chars) ---
REPUBLIQUE TUNISIENNE

## CODE DE L'AMENAGEMENT DU TERRITOIRE ET DE L'URBANISME et ses textes d'application

Publications de l'Imprimerie Officielle de la République Tunisienne 2011

## SOMMAIRE

## CODE DE L'AMENAGEMENT DU TERRITOIRE ET DE L'URBANISME

| SUJET  ARTICLES  PAGE  LOI DE PROMULGATION  1 et 2  7  Code  de  l'aménagement  du  territoire  et  de  l'urbanisme  ………………………….................  1-89  9  TITRE  PREMIER  -  DE  L'AMENAGEMENT  DU  TERRITOIRE…………………………………..  2-11  11  Chapitre  I  -  Du  comité  interministériel  pour  l'aménagement du territoire…………………….  3 et 4  11  Chapitre II - Des schémas directeurs d'aménagement.…...  5-9  12  Chapitre  III  -  Du  suivi  de  la  mise  en  œuvre  des  dispositions  relatives  à  l'aménagement  du  territoire………………………………………...  10 et 11 bis  15  TITRE II - DES PLANS D'AMENAGEMENT URBAIN  12-29  19  Chapitre I - De la limitation des zones requérant l'établissement  ou la révision d'un

### Test Cleaning Logic (Regex)

In [173]:
def bulk_table_cleaner(match):
    line = match.group(0)
    if re.match(r'^[\s|:-]+$', line):
        return re.sub(r'-+', '---', line)
    cells = [cell.strip() for cell in line.split('|')]
    return "| " + " | ".join(c for c in cells if c != "") + " |"

In [174]:
# 1. Setup Patterns
summary_pattern = r"^.*(?:\.{4,}|—{5,}).*$"  # Matches lines with .... or —————

unneeded_phrases = [
    r"Publications de l'Imprimerie Officielle de la République Tunisienne \d{4}",
    r"Sommaire",
    r"Journal Officiel de la République Tunisienne",
    r"Jort\s*:\s*\d{1,4}\s*du\s*\d{1,2}\s*[A-Za-zéû]+\s*\d{4}",
    r"Imprimerie Officielle de la République Tunisienne"
]
unneeded_pattern = '|'.join(unneeded_phrases)

# --- EXECUTE BULK CLEANING ---

# Step 1: Remove Summary lines (TOC), Phrases, Tatweel, and Images
# We use re.MULTILINE so ^ and $ target individual lines in the big text block
md_text = re.sub(summary_pattern, '', md_text, flags=re.MULTILINE)
md_text = re.sub(unneeded_pattern, '', md_text, flags=re.IGNORECASE)
md_text = re.sub(r'/?tatweel(/tatweel)* | /cuspopen*', '', md_text)
md_text = re.sub(r'\u0640', '', md_text)
md_text = re.sub(r'<!-- image -->', '', md_text, flags=re.IGNORECASE)

# Step 2: Inject Newlines and Headers (the ## logic)
keywords_pattern = r"(?<!^)(?<!\n)(?<=\s|[:.])\s*(?=(?:(?i:Article|Art\.)|TITRE|CHAPITRE|SECTION|LIVRE)\b)"
md_text = re.sub(keywords_pattern, r"\n## ", md_text)
md_text = re.sub(r'^##\s*$', '', md_text, flags=re.MULTILINE)

# Step 3: Table Compression
md_text = re.sub(r'^\|.*\|$', bulk_table_cleaner, md_text, flags=re.MULTILINE)

# Step 4: Final Whitespace Polish
# Remove empty lines created by removals and fix spacing
md_text = re.sub(r'^[^\S\r\n]+|[^\S\r\n]+$', '', md_text, flags=re.MULTILINE) # Trim line ends
md_text = re.sub(r'[^\S\r\n]{2,}', ' ', md_text) # Collapse double horizontal spaces
md_text = re.sub(r'\n{2,}', '\n', md_text) # Collapse multiple newlines into single ones for the split

cleaned_markdown = md_text.strip().split('\n')

print(f"✅ Bulk Processing Complete. {len(cleaned_markdown)} lines.")
print ("--- CLEANED MARKDOWN PREVIEW ---")
print(md_text[:1000])

✅ Bulk Processing Complete. 18 lines.
--- CLEANED MARKDOWN PREVIEW ---
REPUBLIQUE TUNISIENNE
## CODE DE L'AMENAGEMENT DU TERRITOIRE ET DE L'URBANISME et ses textes d'application
## CODE DE L'AMENAGEMENT DU TERRITOIRE ET DE L'URBANISME
|---|
|---|
|---|
## Loi n° 94-122 du 28 novembre 1994, portant promulgation du code de l'aménagement du territoire et de l'urbanisme.( )
(JORT n° 96 du 6 décembre 199 4 )
Au nom du peuple, La Chambre des Députés ayant adopté,
Le Président de la République promulgue la loi dont la teneur suit :
## Article premier
Sont promulgués en vertu de la présente loi, sous le titre "code de l'aménagement du territoire et de l'urbanisme" les textes législatifs relatifs à l'aménagement du territoire et à l'urbanisme.
## Article 2
Sont abrogées toutes dispositions antérieures contraires à la présente loi et notamment la loi n° 76-34 du 4 février 1976 relative aux permis de construire, et la loi n° 79-43 du 15 août 1979 portant promulgation du code de l'urbanisme ensemb

In [175]:
clean_lines = md_text.strip().split('\n')

### Test ID and Hierarchy Extraction

In [176]:
def extract_universal_id(line):
    # --- YOUR ORIGINAL CODE (UNCHANGED) ---
    clean = re.sub(r'[#*]', '', line).strip()
    art_match = re.match(r'^(Article|Art\.)\s+(premier|\d+(?:\s*[a-z]+)?)', clean, re.I)
    if art_match:
        val = art_match.group(2).lower()
        return "1" if val == "premier" else val
    
    num_match = re.match(r'^([IVX]+|\d+(?:\.\d+)*)\.?\s+', clean, re.I)
    if num_match:
        return num_match.group(1).strip('.')

    # --- ADDITION: INLINE FALLBACK ---
    # This part handles cases where 'Article X' is buried in a long string.
    # We use a Lookahead (?=[A-Z]) to avoid capturing references like "selon l'article 5".
    inline_pattern = r"(Article\s+(?:premier|\d+(?:\s*(?:bis|ter|quater))?))\s*[\.\-]\s*(?=[A-Z])"
    inline_match = re.search(inline_pattern, clean, re.I)
    
    if inline_match:
        # Extract ID from the first found inline article
        # We look at group 1 but clean it up to match your style
        raw_val = inline_match.group(1).lower()
        # Extract just the "number" part from the label
        id_only = re.sub(r'article\s+', '', raw_val)
        return "1" if id_only == "premier" else id_only.replace(' ', '-')

    return None

# --- UPDATED DETECTION TEST ---
print("--- DETECTION TEST (HYBRID) ---")
for line in clean_lines[:50]:
    new_id = extract_universal_id(line)
    is_hierarchy = re.match(r'^(#+\s*|TITRE|CHAPITRE|SECTION|LIVRE)', line, re.I)
    
    if new_id:
        print(f"🆔 Found ID: {new_id} in line: {line[:60]}...")
    elif is_hierarchy:
        print(f"📂 Found Hierarchy: {line[:60]}...")

--- DETECTION TEST (HYBRID) ---
📂 Found Hierarchy: ## CODE DE L'AMENAGEMENT DU TERRITOIRE ET DE L'URBANISME et ...
📂 Found Hierarchy: ## CODE DE L'AMENAGEMENT DU TERRITOIRE ET DE L'URBANISME...
📂 Found Hierarchy: ## Loi n° 94-122 du 28 novembre 1994, portant promulgation d...
🆔 Found ID: 1 in line: ## Article premier...
🆔 Found ID: 2 in line: ## Article 2...
📂 Found Hierarchy: ## Zine El Abidine Ben Ali...


### Final Payload Construction (The Result)

In [177]:
hierarchy = {"lvl1": "Général", "lvl2": "", "lvl3": ""}
chunks = []
current_chunk_text = []
current_id = "Preamble"

for line in clean_lines:
    new_id = extract_universal_id(line)
    
    if new_id:
        if current_chunk_text:
            # Create payload for previous section
            payload = {
                    "id": f"{LAW_NAME}_{new_id}",
                    "article_number": new_id,
                    "text": "\n".join(current_chunk_text),
                    "metadata": {
                        "source": LAW_NAME.replace('-', ' ').title(),
                    }
                }
            chunks.append(payload)
        
        current_id = new_id
        current_chunk_text = [line]
        continue

    if re.match(r'^(#+\s*|TITRE|CHAPITRE|SECTION|LIVRE)', line, re.I):
        hierarchy["lvl1"] = line
        continue

    current_chunk_text.append(line)

print(f"🎉 Successfully created {len(chunks)} chunks.")
print("--- EXAMPLE CHUNK ---")
if chunks:
    print(json.dumps(chunks[0], indent=4, ensure_ascii=False))
with open(f"{LAW_NAME}_chunks.json", "w", encoding='utf-8') as f:
    json.dump(chunks, f, indent=4, ensure_ascii=False)    


🎉 Successfully created 2 chunks.
--- EXAMPLE CHUNK ---
{
    "id": "code_urbanisme_1",
    "article_number": "1",
    "text": "REPUBLIQUE TUNISIENNE\n|---|\n|---|\n|---|\n(JORT n° 96 du 6 décembre 199 4 )\nAu nom du peuple, La Chambre des Députés ayant adopté,\nLe Président de la République promulgue la loi dont la teneur suit :",
    "metadata": {
        "source": "Code_Urbanisme"
    }
}


## SCRAPING FROM WEB (9anoun.tn)

In [52]:
# 2. Configuration
base_urls = {
    "code-obligations-contrats": "https://9anoun.tn/fr/kb/codes/code-obligations-contrats/code-obligations-contrats-article-",
    "code-droits-reels": "https://9anoun.tn/fr/kb/codes/code-droits-reels/code-droits-reels-article-",
    "code-amenagement-territoire-urbanisme": "https://9anoun.tn/fr/kb/codes/code-amenagement-territoire-urbanisme/code-amenagement-territoire-urbanisme-article-"   
}

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"}
OUTPUT_FILE = "scraped_legal_data.jsonl"
BATCH_SIZE = 10  # Save to file every 10 articles

print ("🚀 Scraper Initialized")


🚀 Scraper Initialized


In [53]:
def extract_clean_article(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    target_div = soup.find("div", class_=lambda x: x and 'prose' in x)
    
    if not target_div:
        return None

    # Remove tooltips
    for tooltip_content in target_div.find_all("span", class_="hs-tooltip-content"):
        tooltip_content.decompose()
        
    text = target_div.get_text(separator=" ", strip=True)
    return " ".join(text.split())

print ("✅ Helper functions defined.")

✅ Helper functions defined.


### Extraction Logic (The Cleaner)
This function targets the specific div and removes nested tooltip metadata so you get clean legal prose.

In [54]:
def extract_clean_article(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    
    # Target the specific container classes
    # We use a CSS selector to find the div that contains 'prose'
    target_div = soup.find("div", class_=lambda x: x and 'prose' in x)
    
    if not target_div:
        return None

    # --- CLEANING STEP ---
    # The tooltips contain definitions (like "Objet: Le matériau sur lequel...")
    # we need to remove the tooltip content span so it doesn't merge with the text.
    for tooltip_content in target_div.find_all("span", class_="hs-tooltip-content"):
        tooltip_content.decompose() # Deletes this element from the tree
        
    # Get text, using a space separator for <br> and other block elements
    text = target_div.get_text(separator=" ", strip=True)
    
    # Remove multiple spaces caused by the cleaning
    text = " ".join(text.split())
    
    return text

print("✅ Extraction function ready.")

✅ Extraction function ready.


### Main Scraper Loop
This cell iterates through the laws, handles the "premier" suffix logic, and stops when a page is not found (404).

In [55]:
def save_batch(buffer, filename):
    """Appends a list of dictionaries to a JSONL file."""
    with open(filename, "a", encoding="utf-8") as f:
        for entry in buffer:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
    logger.info(f"💾 Batch Saved: {len(buffer)} articles written to {filename}")

In [57]:
# --- MAIN EXECUTION ---
batch_buffer = []
total_scraped = 0

# Clear file if starting fresh (optional)
with open(OUTPUT_FILE, "w", encoding="utf-8") as f: pass

for law_id, base_url in base_urls.items():
    logger.info(f"📂 Starting Law: {law_id}")
    i = 1
    consecutive_failures = 0
    
    while True:
        # Handle "premier" vs numbers
        suffix = "premier" if (law_id == "code-amenagement-territoire-urbanisme" and i == 1) else str(i)
        url = f"{base_url}{suffix}"
        
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            
            if response.status_code != 200:
                logger.info(f"⏹️ End of {law_id} reached at Article {i-1}")
                break
                
            clean_text = extract_clean_article(response.content)
            
            if clean_text:
                # Create the requested JSON Payload
                payload = {
                    "id": f"{law_id}_{suffix}",
                    "article_number": suffix,
                    "text": clean_text,
                    "metadata": {
                        "source": law_id.replace('-', ' ').title(),
                    }
                }
                
                batch_buffer.append(payload)
                total_scraped += 1
                consecutive_failures = 0
                
                # Batch processing logic
                if len(batch_buffer) >= BATCH_SIZE:
                    save_batch(batch_buffer, OUTPUT_FILE)
                    batch_buffer = [] # Clear buffer
                    
            else:
                logger.warning(f"⚠️ No content found for {url}")
                consecutive_failures += 1

        except Exception as e:
            logger.error(f"❌ Connection error at {url}: {str(e)}")
            consecutive_failures += 1
            
        if consecutive_failures > 3:
            logger.error(f"🛑 Critical failure for {law_id}. Moving to next law.")
            break
            
        i += 1
        time.sleep(0.3) # Rate limiting

# Final save for any remaining items in buffer
if batch_buffer:
    save_batch(batch_buffer, OUTPUT_FILE)

logger.info(f"✨ ALL DONE. Total articles scraped: {total_scraped}")

2026-04-19 13:40:14,824 - INFO - 📂 Starting Law: code-obligations-contrats
2026-04-19 13:40:33,219 - INFO - 💾 Batch Saved: 10 articles written to scraped_legal_data.jsonl
2026-04-19 13:40:52,136 - INFO - 💾 Batch Saved: 10 articles written to scraped_legal_data.jsonl
2026-04-19 13:41:10,695 - INFO - 💾 Batch Saved: 10 articles written to scraped_legal_data.jsonl
2026-04-19 13:41:30,635 - INFO - 💾 Batch Saved: 10 articles written to scraped_legal_data.jsonl
2026-04-19 13:41:49,783 - INFO - 💾 Batch Saved: 10 articles written to scraped_legal_data.jsonl
2026-04-19 13:42:09,655 - INFO - 💾 Batch Saved: 10 articles written to scraped_legal_data.jsonl
2026-04-19 13:42:29,847 - INFO - 💾 Batch Saved: 10 articles written to scraped_legal_data.jsonl
2026-04-19 13:42:49,498 - INFO - 💾 Batch Saved: 10 articles written to scraped_legal_data.jsonl
2026-04-19 13:43:08,671 - INFO - 💾 Batch Saved: 10 articles written to scraped_legal_data.jsonl
2026-04-19 13:43:28,357 - INFO - 💾 Batch Saved: 10 articles w

In [ ]:
# Preview the first few results

df = pd.DataFrame(all_results)
display(df.head())

# Save to JSONL (standard for LLM training/RAG)
output_file = "scraped_laws.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for entry in all_results:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"💾 Saved data to {output_file}")